# 🛣️ Weighted Shortest Paths — Runnable Notebook

Companion to [`README.md`](README.md) and
[`08_weighted_shortest_paths_lesson.html`](08_weighted_shortest_paths_lesson.html).

**Dijkstra** (non-negative weights, fast) and **Bellman-Ford** (handles negatives, detects negative cycles).

## 1. Why fewest edges ≠ cheapest
BFS counts edges; on a weighted graph we need total **cost**.

In [ ]:
from collections import deque

# Directed weighted graph. adj[u] = list of (v, weight).
#   A=0 B=1 C=2 D=3 E=4
adjw = {
    0: [(1, 1), (2, 4)],       # A->B(1), A->C(4)
    1: [(2, 2), (3, 5)],       # B->C(2), B->D(5)
    2: [(3, 1)],               # C->D(1)
    3: [(4, 3)],               # D->E(3)
    4: [],
}

def bfs_hops(adj, start):
    """Unweighted BFS: distance in EDGES (ignores weights)."""
    dist = {start: 0}; q = deque([start])
    while q:
        u = q.popleft()
        for v, _w in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

hops = bfs_hops(adjw, 0)
print("BFS hop-count to C:", hops[2], "edge (the direct A->C)")
print("...but A->B->C costs 1+2 = 3, cheaper than the direct edge's 4!")
# BFS would pick the 1-edge route (cost 4); the CHEAPEST route costs 3.

## 2. Dijkstra — greedy, min-heap, non-negative weights

In [ ]:
import heapq

def dijkstra(adj, start):
    """Shortest COST from start to every vertex (needs NON-NEGATIVE weights).
       Once a vertex is popped it is FINALISED and never reconsidered -- that is
       what keeps it fast, and also what makes it wrong on negative edges (next cell)."""
    dist = {start: 0}
    done = set()
    pq = [(0, start)]                      # (distance so far, vertex) -- min-heap
    while pq:
        d, u = heapq.heappop(pq)           # the closest unfinished vertex
        if u in done:
            continue                       # already finalised -> stale entry, skip
        done.add(u)                        # LOCK u: its distance is now final
        for v, w in adj[u]:
            if v in done:
                continue                   # never relax an already-finalised vertex
            nd = d + w                      # cost to reach v THROUGH u
            if nd < dist.get(v, float("inf")):
                dist[v] = nd                # relax: cheaper path found
                heapq.heappush(pq, (nd, v))
    return dist

dist = dijkstra(adjw, 0)
print("Dijkstra distances from A:", dist)
assert dist == {0: 0, 1: 1, 2: 3, 3: 4, 4: 7}   # note C=3 (via B), not 4

## 3. Why negative edges break Dijkstra
Dijkstra **locks** a vertex when it's popped and never reconsiders it — a later negative edge can invalidate that.

In [ ]:
# 0->1 (4), 0->2 (5), 2->1 (-3): the cheapest route to 1 is 0->2->1 = 5 + (-3) = 2
neg_adj = {0: [(1, 4), (2, 5)], 1: [], 2: [(1, -3)]}
wrong = dijkstra(neg_adj, 0)[1]
print("Dijkstra says dist[1] =", wrong, " (WRONG: the true shortest is 2)")
# Dijkstra finalises vertex 1 at distance 4 the moment it is popped, so the later
# 2->1 (-3) shortcut is never applied. Bellman-Ford (next) gets it right.
assert wrong == 4                          # the textbook failure on a negative edge

## 4. Bellman-Ford — relax every edge V−1 times (negatives OK)

In [ ]:
def bellman_ford(vertices, edges, start):
    """edges = list of (u, v, w). Returns (dist, has_negative_cycle)."""
    dist = {v: float("inf") for v in vertices}
    dist[start] = 0
    for _ in range(len(vertices) - 1):     # V-1 rounds
        for u, v, w in edges:
            if dist[u] + w < dist[v]:       # relax every edge each round
                dist[v] = dist[u] + w
    # extra pass: if anything still improves, a negative cycle is reachable
    neg = any(dist[u] + w < dist[v] for u, v, w in edges)
    return dist, neg

edges = [(0, 1, 4), (0, 2, 5), (2, 1, -3)]
dist, neg = bellman_ford([0, 1, 2], edges, 0)
print("Bellman-Ford dist:", dist, "| negative cycle?", neg)
assert dist[1] == 2 and not neg                 # correct: 1 is reached at cost 2

## 5. Detecting a negative cycle

In [ ]:
# 0->1 (1), 1->2 (-1), 2->0 (-1): the loop sums to -1, so costs can shrink forever
cyc_edges = [(0, 1, 1), (1, 2, -1), (2, 0, -1)]
_, neg = bellman_ford([0, 1, 2], cyc_edges, 0)
print("negative cycle detected?", neg)
assert neg is True

## ✅ Recap
- **Relaxation** `if dist[u]+w < dist[v]: dist[v] = dist[u]+w` is the shared core.
- **Dijkstra**: greedy + min-heap, **non-negative only**, `O((V+E) log V)`.
- **Bellman-Ford**: relax all edges `V−1×`, handles negatives, **detects negative cycles**, `O(V·E)`.
- Unweighted? plain **BFS**. All-pairs? **Floyd-Warshall** `O(V³)`.

Next: [`09_Minimum_Spanning_Tree`](../09_Minimum_Spanning_Tree/README.md).